# When a bird map looks empty: occurrence records are not absence

**Jihun Oh | EDS Book idea #14 | preparation draft, NOT a peer-reviewed or Binder-tested notebook.**

## Purpose
Compare analyst-chosen spatial grids and explain how quality criteria affect mapped *occurrence records*. KoreAves is the author's motivation, **not** the underlying source dataset. The method is transferable to other countries after independent local checks.

**Learning objectives:** Distinguish reported location uncertainty from grid size; count records rather than birds; see that empty cells cannot prove a species absent; disclose source and sampling limitations.


## Data and attribution
The live GBIF occurrence search API is used for South Korean human bird observations in **2024**, constrained to CC0 records with coordinates. This query takes only the **first 100 API results**. It is **non-random**, **mutable**, not a time series, and **has no occurrence-download DOI**. It is an *illustrative preview*, not a fixed citable sample for publication. Original datasetKey values remain visible for attribution; data and imagery licences may differ. Before review, use a stable, citable [GBIF download DOI](https://www.gbif.org/citation-guidelines). No third-party media are copied.


In [ ]:
from collections import Counter
from urllib.parse import urlencode
from urllib.request import Request,urlopen
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pyproj import Transformer
from occurrence_grid import screen,aggregate
query={'country':'KR','taxonKey':'212','basisOfRecord':'HUMAN_OBSERVATION','year':'2024','license':'CC0_1_0','hasCoordinate':'true','hasGeospatialIssue':'false','limit':'100','offset':'0'}
url='https://api.gbif.org/v1/occurrence/search?'+urlencode(query)
with urlopen(Request(url,headers={'User-Agent':'EDSBook-records-education/0.1'}),timeout=30) as response:
    result=json.load(response)
records=result['results']
assert 0<len(records)<=100
print('Live GBIF first-page records:',len(records),'of search matches:',result.get('count'))
print('Exact live query:',url)
print('Dataset keys for independent publisher citation:',dict(Counter(str(x.get('datasetKey')) for x in records)))


## Quality screening
Exclude missing/invalid coordinate uncertainty and uncertainty values above **1,000 metres**, an *illustrative exercise threshold*, NOT a biodiversity quality standard. Values missing from the metadata do **not** imply perfect accuracy. Duplicate GBIF IDs are counted once. The selection still has taxonomic, access and observer-effort bias.


In [ ]:
accepted,exclusions=screen(records,threshold=1000)
print('Accepted records:',len(accepted))
print('Exclusions by first failing rule:',dict(sorted(exclusions.items())))
assert len(accepted)+sum(exclusions.values())==len(records)
print('Accepted dataset keys:',dict(Counter(x['datasetKey'] for x in accepted)))


## Map two spatial aggregation scales
Reproject longitude/latitude (WGS84) into **WGS84 / UTM zone 52N**, a metric projection appropriate for this *illustrative* Korean example. UTM zone selection should be reassessed outside the study area. **10 km** and **50 km** are analyst-defined bin sizes, not the input data's native precision. A blank grid cell is a lack of qualifying *records* under these rules?not evidence of biological absence.


In [ ]:
if not accepted:
    raise ValueError('No qualifying records in this API first page; do not invent records or silently lower quality criteria')
transform=Transformer.from_crs('EPSG:4326','EPSG:32652',always_xy=True)
xy=[transform.transform(x['lon'],x['lat']) for x in accepted]
fine=aggregate(xy,10000)
coarse=aggregate(xy,50000)
assert sum(fine.values())==sum(coarse.values())==len(accepted)
print('10-km occupied cells:',len(fine),'50-km occupied cells:',len(coarse),'accepted RECORDS:',len(accepted))


In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,5),constrained_layout=True)
for ax,(title,step,counts) in zip(axs,[('10 km',10000,fine),('50 km',50000,coarse)]):
    for (gx,gy),count in counts.items():
        ax.add_patch(Rectangle((gx*step/1000,gy*step/1000),step/1000,step/1000,alpha=0.5,edgecolor='black'))
    ax.autoscale_view()
    ax.set_aspect('equal',adjustable='datalim')
    ax.set_title(title+': '+str(len(counts))+' occupied cells')
    ax.set_xlabel('UTM 52N easting, km')
    ax.set_ylabel('UTM 52N northing, km')
    ax.grid(True,alpha=0.2)
plt.suptitle('Accepted RECORD counts, not bird abundance or absence')
plt.show()


## Interpret: claim, evidence, uncertainty

**Supported:** For the current non-random API first page and declared exclusions, an occupied cell contains at least one accepted observation record.

**Unsupported:** A blank cell proves the bird absent; more records show greater bird abundance; this map predicts where a visitor will see a bird; different cell counts constitute a population trend.

**What would strengthen such claims?** Citable versioned primary source, accurate taxon matching, appropriate sampling frame, surveyed zeroes, observer-effort information, seasonal context, and independently confirmed public site access. The 10-km/50-km view does not resolve those issues.

**Exercise:** Change one analyst grid size. State one visual inference that changes, one that remains valid, and one that is unsupported in both maps.


## Summary and next steps

- Mapped accepted *records* are not birds, ranges or biological absences.
- Missing uncertainty is unknown and is disclosed as an exclusion.
- Grid size and source-coordinate uncertainty are different quantities.
- Before sharing this as an EDS Book-reviewed notebook: create a versioned, DOI-citable occurrence dataset; reconcile all publisher credits/licences; rerun across the intended geography; test on Binder; send the editor the PR proof.

**Original data access:** [GBIF](https://www.gbif.org/citation-guidelines). **Code:** Python; upstream official EDS Book template retained. **Contact:** comments in [idea #14](https://github.com/eds-book/notebooks-ideas/issues/14).
